# LeetCode 322 — Coin Change bằng Python

Link đề: https://leetcode.com/problems/coin-change/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming, BFS, Array

Skeleton LeetCode:

```python
class Solution:
    def coinChange(self, coins: List[int], amount: int) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (liệt kê mọi bộ số lượng từng loại xu).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho các loại xu (mỗi loại **dùng không giới hạn**) và một tổng tiền `amount`, tìm **số xu ít nhất** để tạo đúng `amount`; không tạo được thì trả `-1`.
2. Hiểu vì sao **tham lam (greedy)** "luôn lấy xu lớn nhất" **sai** với hệ xu bất kỳ, nên cần **quy hoạch động (DP)**.
3. Đi từ **vét cạn đệ quy** (hàm mũ) → **đệ quy có nhớ (top-down)** `O(amount · n)` → **DP bottom-up** `dp[a] = 1 + min(dp[a - c])` → **BFS theo số xu** (mỗi lớp = thêm một xu) → **DP có sắp xếp xu và cắt tỉa**.
4. Nắm khung DP **"đủ tiền thối"** (unbounded knapsack): trạng thái là **số tiền còn lại**, chuyển trạng thái là **chọn một loại xu cuối cùng**.
5. Tránh các lỗi kinh điển: **greedy**, khởi tạo `dp` bằng `0`, quên đổi `inf` thành `-1`, xu **lớn hơn** số tiền (chỉ số âm), và nhầm **số cách** (Coin Change II) với **số xu ít nhất**.

## 2. Đề bài & ví dụ

Cho mảng `coins` là các mệnh giá xu khác nhau và một số nguyên `amount` là tổng tiền.

Trả về **số xu ít nhất** cần để tạo đủ `amount`. Nếu **không thể** tạo được `amount` bằng bất kỳ tổ hợp xu nào thì trả `-1`.

Giả sử bạn có **số lượng vô hạn** mỗi loại xu.

**Ví dụ 1**

- Input: `coins = [1,2,5], amount = 11`
- Output: `3`
- Giải thích: `11 = 5 + 5 + 1`.

**Ví dụ 2**

- Input: `coins = [2], amount = 3`
- Output: `-1`

**Ví dụ 3**

- Input: `coins = [1], amount = 0`
- Output: `0`

**Ràng buộc:** `1 <= coins.length <= 12`; `1 <= coins[i] <= 2^31 - 1`; `0 <= amount <= 10^4`.

In [1]:
from typing import List
from collections import deque
from functools import lru_cache
from itertools import product
import random
import sys
import time

examples = [
    ([1, 2, 5], 11, 3),
    ([2], 3, -1),
    ([1], 0, 0),
]
for coins, amount, expected in examples:
    print(f"coins={coins}, amount={amount} -> kỳ vọng {expected}")

coins=[1, 2, 5], amount=11 -> kỳ vọng 3
coins=[2], amount=3 -> kỳ vọng -1
coins=[1], amount=0 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Vô cực `float("inf")` làm "chưa tạo được"

Trong DP tìm **giá trị nhỏ nhất**, ta khởi tạo các ô chưa biết bằng `float("inf")` (lớn hơn mọi số). `min(inf, x)` luôn là `x`; `inf + 1` vẫn là `inf`. Cuối cùng nếu đáp án vẫn `inf` thì không tạo được ⇒ trả `-1`.
#### Code demo

In [2]:
INF = float("inf")
print(min(INF, 5), INF + 1, INF == INF + 1, 5 < INF)       # 5 inf True True

5 inf True True


### 3.2. Vì sao tham lam sai

Với `coins = [1, 3, 4]`, `amount = 6`: tham lam lấy `4` trước rồi còn `2 = 1 + 1` ⇒ `3` xu (`4 + 1 + 1`); nhưng tối ưu là `3 + 3` chỉ `2` xu. Tham lam chỉ đúng với một số hệ xu đặc biệt (như `1, 5, 10, 25`).
#### Code demo

In [3]:
def greedy(coins, amount):
    count = 0
    for c in sorted(coins, reverse=True):
        count += amount // c
        amount %= c
    return count if amount == 0 else -1


print("tham lam [1,3,4], 6:", greedy([1, 3, 4], 6), "| tối ưu: 2 (3 + 3)")

tham lam [1,3,4], 6: 3 | tối ưu: 2 (3 + 3)


### 3.3. Hệ thức quy hoạch động

Gọi `dp[a]` là **số xu ít nhất để tạo đúng `a`**. Xu **cuối cùng** có thể là bất kỳ `c ∈ coins` với `c <= a`; trước đó đã tạo `a - c`. Vậy `dp[a] = 1 + min(dp[a - c] : c <= a)` và `dp[0] = 0`.
#### Code demo

In [4]:
coins, amount = [1, 2, 5], 11
dp = [0] + [INF] * amount
for a in range(1, amount + 1):
    for c in coins:
        if c <= a and dp[a - c] + 1 < dp[a]:
            dp[a] = dp[a - c] + 1
print(dp)            # [0, 1, 1, 2, 2, 1, 2, 2, 3, 3, 2, 3]
print("dp[11] =", dp[11])

[0, 1, 1, 2, 2, 1, 2, 2, 3, 3, 2, 3]
dp[11] = 3


### 3.4. Đệ quy có nhớ (`lru_cache`)

`@lru_cache` nhớ kết quả hàm theo đối số. `f(a)` = số xu ít nhất tạo `a`; gọi `f(a - c)` cho từng xu; mỗi `a` chỉ tính một lần ⇒ `O(amount · n)`. Lưu ý: độ sâu đệ quy có thể tới `amount / min(coins)` (tối đa `10^4`) nên cần nâng giới hạn đệ quy.
#### Code demo

In [5]:
calls = 0


@lru_cache(maxsize=None)
def f(a):
    global calls
    calls += 1
    if a == 0:
        return 0
    return min((f(a - c) + 1 for c in (1, 2, 5) if c <= a), default=INF)


print(f(11), "số lần tính thật sự:", calls)       # 3, 12 (mỗi a từ 0 đến 11 một lần)

3 số lần tính thật sự: 12


### 3.5. BFS trên "số tiền còn lại"

Coi mỗi số tiền `a` là một **nút**; thêm một xu `c` là đi từ `a` tới `a + c` (hoặc từ `a` tới `a - c`). **Số xu ít nhất** = **khoảng cách ngắn nhất** từ `0` tới `amount` trong đồ thị không trọng số ⇒ **BFS** (mỗi lớp là thêm một xu).
#### Code demo

In [6]:
level, frontier, seen = 0, {0}, {0}
while frontier and 11 not in frontier:
    level += 1
    frontier = {a + c for a in frontier for c in (1, 2, 5) if a + c <= 11 and a + c not in seen}
    seen |= frontier
    print(f"{level} xu: các tổng mới đạt được:", sorted(frontier))
print("số xu ít nhất:", level)

1 xu: các tổng mới đạt được: [1, 2, 5]
2 xu: các tổng mới đạt được: [3, 4, 6, 7, 10]
3 xu: các tổng mới đạt được: [8, 9, 11]
số xu ít nhất: 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`coins = [1, 2, 5]`, `amount = 11`:

- `dp[0] = 0`.
- `dp[1] = 1` (xu `1`), `dp[2] = 1` (xu `2`), `dp[3] = 2` (`1+2`), `dp[4] = 2` (`2+2`), `dp[5] = 1` (xu `5`).
- `dp[6] = 1 + min(dp[5], dp[4], dp[1]) = 2`; `dp[7] = 2` (`5+2`); `dp[8] = 3` (`5+2+1`)...
- `dp[11] = 1 + min(dp[10], dp[9], dp[6]) = 1 + min(2, 3, 2) = 3`.

**Cách nghĩ:** xét **xu cuối cùng** ta thêm vào; phần còn lại là bài toán nhỏ hơn cùng dạng ⇒ lưu đáp án các tổng nhỏ rồi dùng lại.

## 5. Cách 1 — Brute Force / baseline: đệ quy thử mọi xu, không nhớ

**Ý tưởng.** `solve(a)`: nếu `a == 0` ⇒ `0`; nếu `a < 0` ⇒ vô hạn; ngược lại `1 + min(solve(a - c))` qua mọi xu. Không nhớ ⇒ tính lại rất nhiều lần.
#### Code demo

In [7]:
def coin_change_bruteforce(coins, amount):
    def solve(a):
        if a == 0:
            return 0
        best = INF
        for c in coins:
            if c <= a:
                best = min(best, solve(a - c) + 1)
        return best

    ans = solve(amount)
    return -1 if ans == INF else ans


for coins, amount, expected in examples:
    print(coins, amount, "->", coin_change_bruteforce(coins, amount), "(kỳ vọng", expected, ")")

[1, 2, 5] 11 -> 3 (kỳ vọng 3 )
[2] 3 -> -1 (kỳ vọng -1 )
[1] 0 -> 0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Gọi `S = amount`, `n = len(coins)`. Cây đệ quy có tới `n^S` nhánh (xu nhỏ làm độ sâu `S`): **Time `O(n^S)`** — bùng nổ (với `amount = 30` đã rất chậm); **Space `O(S)`**.

**Nút thắt.** Cùng một số tiền còn lại `a` được tính lại vô số lần. Chỉ có `amount + 1` giá trị `a` khác nhau.

## 6. Ý tưởng thuật toán / bước đột phá

**Bài toán con trùng lặp** ⇒ **ghi nhớ**: tối đa `amount + 1` trạng thái, mỗi trạng thái xét `n` xu ⇒ **`O(amount · n)`**. Có thể làm từ trên xuống (đệ quy + nhớ) hoặc từ dưới lên (điền bảng `dp[0..amount]`).

Cách nhìn khác: **BFS** theo số xu — lớp `k` là mọi tổng tạo được bằng đúng `k` xu; lớp đầu tiên chứa `amount` cho đáp án.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (top-down)

**Giải thích.** Như Cách 1 nhưng thêm từ điển `memo` (có thể dùng `lru_cache`, nhưng với độ sâu tới `10^4` thì từ điển tự quản an toàn hơn).
#### Code demo

In [8]:
def coin_change_memo(coins, amount):
    sys.setrecursionlimit(max(sys.getrecursionlimit(), 30_000))
    memo = {0: 0}                                         # từ điển tự quản (thay lru_cache để tránh giới hạn đệ quy mức C)

    def solve(a):
        if a in memo:
            return memo[a]
        best = INF
        for c in coins:
            if c <= a:
                best = min(best, solve(a - c) + 1)
        memo[a] = best
        return best

    ans = solve(amount)
    return -1 if ans == INF else ans


for coins, amount, expected in examples:
    print(coins, amount, "->", coin_change_memo(tuple(coins), amount), "(kỳ vọng", expected, ")")

[1, 2, 5] 11 -> 3 (kỳ vọng 3 )
[2] 3 -> -1 (kỳ vọng -1 )
[1] 0 -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 2:** **Time `O(amount · n)`**, **Space `O(amount)`** (bộ nhớ đệm và ngăn xếp đệ quy sâu tới `amount / min(coins)`).

### Cách 3 — DP bottom-up (cách nộp chuẩn)

**Giải thích.** `dp[0] = 0`, `dp[a] = inf`; với `a` từ `1` tới `amount`, với mỗi xu `c <= a`: `dp[a] = min(dp[a], dp[a - c] + 1)`. Đáp án `dp[amount]` (hoặc `-1` nếu `inf`).
#### Code demo

In [9]:
def coin_change_dp(coins, amount):
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
    return -1 if dp[amount] == INF else dp[amount]


for coins, amount, expected in examples:
    print(coins, amount, "->", coin_change_dp(coins, amount), "(kỳ vọng", expected, ")")

[1, 2, 5] 11 -> 3 (kỳ vọng 3 )
[2] 3 -> -1 (kỳ vọng -1 )
[1] 0 -> 0 (kỳ vọng 0 )


In [10]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def coinChange(self, coins: List[int], amount: int) -> int:
        n = amount
        f = [0] + [float("inf")] * n
        for x in coins:
            for j in range(x, n + 1):
                f[j] = min(f[j], f[j - x] + 1)
        return -1 if f[n] == float("inf") else f[n]


print(Solution().coinChange([1, 2, 5], 11))     # 3

3


**Độ phức tạp Cách 3:** **Time `O(amount · n)`** (`10^4 × 12 = 1.2·10^5` phép), **Space `O(amount)`**. Bản nộp ở trên đổi thứ tự hai vòng (duyệt từng loại xu ngoài, tổng trong) — cho cùng kết quả vì mỗi xu dùng được nhiều lần.

### Cách 4 — BFS theo số xu

**Giải thích.** Hàng đợi chứa các tổng đã tạo; mỗi lớp thêm một xu; lần đầu chạm `amount` trả số lớp. Dùng `visited` để mỗi tổng vào hàng đợi một lần.
#### Code demo

In [11]:
def coin_change_bfs(coins, amount):
    if amount == 0:
        return 0
    seen = {0}
    queue = deque([0])
    steps = 0
    while queue:
        steps += 1
        for _ in range(len(queue)):                  # một lớp = thêm đúng một xu
            a = queue.popleft()
            for c in coins:
                nxt = a + c
                if nxt == amount:
                    return steps
                if nxt < amount and nxt not in seen:
                    seen.add(nxt)
                    queue.append(nxt)
    return -1


for coins, amount, expected in examples:
    print(coins, amount, "->", coin_change_bfs(coins, amount), "(kỳ vọng", expected, ")")

[1, 2, 5] 11 -> 3 (kỳ vọng 3 )
[2] 3 -> -1 (kỳ vọng -1 )
[1] 0 -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 4:** **Time `O(amount · n)`** (mỗi tổng vào hàng đợi tối đa một lần, xét `n` xu), **Space `O(amount)`**. Có thể dừng sớm khi gặp `amount`.

### Cách 5 — DP với xu sắp xếp và `break` (cắt tỉa)

**Giải thích.** Sắp xếp xu tăng dần; trong vòng `for c` có thể `break` ngay khi `c > a` (các xu sau còn lớn hơn). Bỏ trước các xu `> amount` (giá trị tới `2^31 - 1`).
#### Code demo

In [12]:
def coin_change_pruned(coins, amount):
    usable = sorted(c for c in set(coins) if c <= amount)
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        best = INF
        for c in usable:
            if c > a:
                break
            if dp[a - c] + 1 < best:
                best = dp[a - c] + 1
        dp[a] = best
    return -1 if dp[amount] == INF else dp[amount]


for coins, amount, expected in examples:
    print(coins, amount, "->", coin_change_pruned(coins, amount), "(kỳ vọng", expected, ")")

[1, 2, 5] 11 -> 3 (kỳ vọng 3 )
[2] 3 -> -1 (kỳ vọng -1 )
[1] 0 -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 5:** **Time `O(amount · n)`** (hằng số nhỏ hơn nhờ `break`), **Space `O(amount)`**.

> Cách 3 là bản nộp chuẩn; Cách 4 cho góc nhìn đồ thị; Cách 2 là dạng từ trên xuống.

## 8. TRACE từng bước

Ta in bảng `dp` (kèm xu cuối cùng được chọn) cho `coins = [1, 2, 5]`, `amount = 11` và các lớp BFS cho `coins = [2, 5]`, `amount = 7`.
#### Code demo

In [13]:
def trace_dp(coins, amount):
    dp = [0] + [INF] * amount
    last = [None] * (amount + 1)
    print(f"coins={coins}, amount={amount}")
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
                last[a] = c
        shown = "inf" if dp[a] == INF else dp[a]
        print(f"  dp[{a:>2}] = {shown}  (xu cuối {last[a]})" + (f"; trước đó dp[{a - last[a]}] = {dp[a - last[a]]}" if last[a] else ""))
    return dp[amount]


trace_dp([1, 2, 5], 11)

coins=[1, 2, 5], amount=11
  dp[ 1] = 1  (xu cuối 1); trước đó dp[0] = 0
  dp[ 2] = 1  (xu cuối 2); trước đó dp[0] = 0
  dp[ 3] = 2  (xu cuối 1); trước đó dp[2] = 1
  dp[ 4] = 2  (xu cuối 2); trước đó dp[2] = 1
  dp[ 5] = 1  (xu cuối 5); trước đó dp[0] = 0
  dp[ 6] = 2  (xu cuối 1); trước đó dp[5] = 1
  dp[ 7] = 2  (xu cuối 2); trước đó dp[5] = 1
  dp[ 8] = 3  (xu cuối 1); trước đó dp[7] = 2
  dp[ 9] = 3  (xu cuối 2); trước đó dp[7] = 2
  dp[10] = 2  (xu cuối 5); trước đó dp[5] = 1
  dp[11] = 3  (xu cuối 1); trước đó dp[10] = 2


3

In [14]:
def trace_bfs(coins, amount):
    seen, frontier, steps = {0}, [0], 0
    print(f"coins={coins}, amount={amount}")
    while frontier:
        steps += 1
        new = []
        for a in frontier:
            for c in coins:
                nxt = a + c
                if nxt == amount:
                    print(f"  lớp {steps}: từ {a} thêm xu {c} tới {amount} -> đáp án {steps}")
                    return steps
                if nxt < amount and nxt not in seen:
                    seen.add(nxt)
                    new.append(nxt)
        print(f"  lớp {steps}: các tổng mới {new}")
        frontier = new
    print("  không tới được amount -> -1")
    return -1


trace_bfs([2, 5], 7)
print()
trace_bfs([2], 3)

coins=[2, 5], amount=7
  lớp 1: các tổng mới [2, 5]
  lớp 2: từ 2 thêm xu 5 tới 7 -> đáp án 2

coins=[2], amount=3
  lớp 1: các tổng mới [2]
  lớp 2: các tổng mới []
  không tới được amount -> -1


-1

Bảng tóm tắt (`[1,2,5]`, `11`): `dp = [0, 1, 1, 2, 2, 1, 2, 2, 3, 3, 2, 3]`; xu cuối cùng của `dp[11]` là `5` hoặc `1` (`dp[6] = 2` nên `1 + 2 = 3`). Đáp án `3`.

## 9. Code nộp LeetCode

Bản nộp dùng DP bottom-up. Đúng `class Solution`, đúng `coinChange(self, coins: List[int], amount: int) -> int`.

In [15]:
class Solution:
    def coinChange(self, coins: List[int], amount: int) -> int:
        dp = [0] + [float("inf")] * amount             # dp[a]: số xu ít nhất tạo đúng a
        for a in range(1, amount + 1):
            for c in coins:
                if c <= a:                              # xu lớn hơn a không dùng được (tránh chỉ số âm)
                    dp[a] = min(dp[a], dp[a - c] + 1)
        return -1 if dp[amount] == float("inf") else dp[amount]


sol = Solution()
for coins, amount, expected in examples:
    assert sol.coinChange(coins, amount) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Cấu trúc con tối ưu:** một cách tạo `a` bằng ít xu nhất kết thúc bằng một xu `c` nào đó; bỏ xu đó đi, phần còn lại phải là cách tối ưu tạo `a - c` (nếu không, thay bằng cách tốt hơn thì tổng số xu giảm). Vậy `dp[a] = 1 + min_c dp[a - c]`.
- **Cơ sở:** `dp[0] = 0` (không dùng xu nào); `dp[a] = inf` nếu không xu nào đưa về trạng thái tạo được.
- **Thứ tự tính:** `dp[a]` chỉ phụ thuộc `dp` của các giá trị **nhỏ hơn** ⇒ điền tăng dần là đủ.
- **Không tạo được:** nếu mọi `dp[a - c] = inf` thì `dp[a] = inf`; `inf` lan truyền đến `dp[amount]` ⇒ `-1`.
- **BFS:** số xu ít nhất bằng khoảng cách ngắn nhất từ `0` tới `amount` trong đồ thị cạnh `a → a + c`; BFS tìm khoảng cách ngắn nhất.
- **Hoán đổi thứ tự vòng lặp** (xu ngoài, tiền trong) vẫn đúng vì ta tính **số xu ít nhất** (không phân biệt thứ tự); chỉ bài **đếm số cách** (Coin Change II) mới phụ thuộc thứ tự vòng lặp.

## 11. Phân tích Time / Space Complexity

Gọi `S = amount` (≤ `10^4`), `n = len(coins)` (≤ `12`).

| Cách | Time | Space |
|---|---|---|
| 1. Đệ quy thuần | `O(n^S)` | `O(S)` |
| 2. Đệ quy có nhớ | `O(S·n)` | `O(S)` |
| 3. DP bottom-up | `O(S·n)` | `O(S)` |
| 4. BFS | `O(S·n)` | `O(S)` |
| 5. DP sắp xếp + `break` | `O(S·n)` | `O(S)` |

Với `S = 10^4`, `n = 12`: cỡ `1.2·10^5` phép — tức thời. Không có khác biệt average/worst về bậc (BFS có thể dừng sớm nên tốt hơn khi đáp án nhỏ).

## 12. Bộ test

`oracle` là brute force độc lập: **liệt kê mọi bộ số lượng** `(k_1, ..., k_n)` với `Σ k_i · c_i == amount` bằng `itertools.product` (chỉ cho `amount` nhỏ), lấy `Σ k_i` nhỏ nhất. Với `amount` lớn đối chiếu các cách DP/BFS với nhau.
#### Code demo

In [16]:
def oracle(coins, amount):
    if amount == 0:
        return 0
    best = None
    ranges = [range(amount // c + 1) for c in coins]
    for counts in product(*ranges):
        if sum(k * c for k, c in zip(counts, coins)) == amount:
            total = sum(counts)
            if best is None or total < best:
                best = total
    return -1 if best is None else best


impls = {
    "bruteforce": coin_change_bruteforce,
    "memo": lambda coins, amount: coin_change_memo(tuple(coins), amount),
    "dp": coin_change_dp,
    "bfs": coin_change_bfs,
    "pruned": coin_change_pruned,
    "submit": sol.coinChange,
}

for coins, amount, expected in examples:
    assert oracle(coins, amount) == expected
    for key, fn in impls.items():
        assert fn(list(coins), amount) == expected, (key, coins, amount, fn(list(coins), amount), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [17]:
edge_cases = [
    ([1], 0, 0, "amount = 0"),
    ([5], 0, 0, "amount = 0, xu lớn"),
    ([1], 7, 7, "chỉ xu 1"),
    ([2], 4, 2, "chia hết"),
    ([2], 3, -1, "không chia hết"),
    ([3, 5], 7, -1, "không tạo được 7 từ 3 và 5"),
    ([3, 5], 8, 2, "3 + 5"),
    ([1, 3, 4], 6, 2, "tham lam sai: 3 + 3"),
    ([2, 5, 10, 1], 27, 4, "10 + 10 + 5 + 2"),
    ([186, 419, 83, 408], 6249, 20, "ca LeetCode quen thuộc"),
    ([2147483647], 2, -1, "xu lớn hơn amount (giá trị biên 2^31 - 1)"),
    ([1, 2147483647], 2, 2, "một xu quá lớn, một xu 1"),
    ([5, 7], 1, -1, "mọi xu lớn hơn amount"),
]
for coins, amount, expected, label in edge_cases:
    if amount <= 40:
        assert oracle([c for c in coins if c <= amount] or [amount + 1], amount) == expected, (label, oracle(coins, amount), expected)
    for key, fn in impls.items():
        if key == "bruteforce" and amount > 14:
            continue
        assert fn(list(coins), amount) == expected, (key, label, fn(list(coins), amount), expected)
    print("PASS", label)

PASS amount = 0
PASS amount = 0, xu lớn
PASS chỉ xu 1
PASS chia hết
PASS không chia hết
PASS không tạo được 7 từ 3 và 5
PASS 3 + 5
PASS tham lam sai: 3 + 3
PASS 10 + 10 + 5 + 2
PASS ca LeetCode quen thuộc
PASS xu lớn hơn amount (giá trị biên 2^31 - 1)
PASS một xu quá lớn, một xu 1
PASS mọi xu lớn hơn amount


### 12.2. Tính chất và phản ví dụ

- Đáp án `-1` hoặc `>= ceil(amount / max(coins))`; nếu `1 ∈ coins` thì luôn tạo được và đáp án `<= amount`.
- **Thêm một loại xu** không bao giờ làm đáp án **tăng** (có thêm lựa chọn).
- **Nhân tất cả mệnh giá và `amount` với `d`** không đổi đáp án.
- **Phản ví dụ** cho tham lam: `[1,3,4]`, `6` cho `3` thay vì `2`.
#### Code demo

In [18]:
rng = random.Random(322)
for _ in range(300):
    coins = rng.sample(range(1, 15), rng.randint(1, 4))
    amount = rng.randint(0, 60)
    ans = sol.coinChange(coins, amount)
    if ans != -1:
        assert ans >= -(-amount // max(coins))
    if 1 in coins:
        assert ans != -1 and ans <= amount
    extra = rng.randint(1, 20)
    ans2 = sol.coinChange(coins + [extra], amount)
    assert ans == -1 or (ans2 != -1 and ans2 <= ans)
    d = rng.randint(2, 5)
    assert sol.coinChange([c * d for c in coins], amount * d) == ans
print("PASS tính chất (cận dưới, có xu 1, thêm xu, nhân tỉ lệ) trên 300 bộ ngẫu nhiên")
print("tham lam [1,3,4], 6:", greedy([1, 3, 4], 6), "| đúng:", sol.coinChange([1, 3, 4], 6))
assert greedy([1, 3, 4], 6) != sol.coinChange([1, 3, 4], 6)

PASS tính chất (cận dưới, có xu 1, thêm xu, nhân tỉ lệ) trên 300 bộ ngẫu nhiên
tham lam [1,3,4], 6: 3 | đúng: 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [19]:
rng = random.Random(4322)
for trial in range(1500):
    coins = rng.sample(range(1, 13), rng.randint(1, 4))
    amount = rng.randint(0, 24)
    truth = oracle(coins, amount)
    for key, fn in impls.items():
        if key == "bruteforce" and amount > 14:
            continue                                   # đệ quy thuần bùng nổ khi amount lớn
        assert fn(list(coins), amount) == truth, (key, coins, amount, fn(list(coins), amount), truth)
print("PASS 1500 bộ ngẫu nhiên nhỏ: mọi cách khớp oracle liệt kê")

# amount = 10^4, 12 loại xu: so các cách nhanh với nhau
cases = [
    (sorted(rng.sample(range(1, 500), 12)), 10_000),
    ([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000], 9_999),
    ([7, 13, 29, 101, 997], 10_000),
    ([5000, 5001], 10_000),
    ([2 ** 31 - 1, 3], 10_000),
]
for coins, amount in cases:
    row, ref = [], None
    for key in ("memo", "dp", "bfs", "pruned", "submit"):
        t0 = time.perf_counter()
        value = impls[key](list(coins), amount)
        dt = (time.perf_counter() - t0) * 1000
        ref = value if ref is None else ref
        assert value == ref, (key, coins, amount)
        row.append(f"{key}={dt:6.1f}ms")
    print(f"amount={amount} coins={coins[:4]}...: đáp án {ref} | " + " ".join(row))
print("PASS amount lớn")

PASS 1500 bộ ngẫu nhiên nhỏ: mọi cách khớp oracle liệt kê


amount=10000 coins=[72, 75, 145, 205]...: đáp án 22 | memo=  14.3ms dp=   6.6ms bfs=   9.2ms pruned=   8.9ms submit=   8.3ms


amount=9999 coins=[1, 2, 5, 10]...: đáp án 12 | memo=  16.0ms dp=   6.2ms bfs=   7.3ms pruned=   6.3ms submit=   8.0ms
amount=10000 coins=[7, 13, 29, 101]...: đáp án 26 | memo=   6.2ms dp=   2.9ms bfs=   3.5ms pruned=   2.7ms submit=   3.2ms
amount=10000 coins=[5000, 5001]...: đáp án 2 | memo=   0.0ms dp=   0.9ms bfs=   0.0ms pruned=   0.9ms submit=   1.1ms
amount=10000 coins=[2147483647, 3]...: đáp án -1 | memo=   1.0ms dp=   1.2ms bfs=   0.7ms pruned=   1.0ms submit=   1.3ms
PASS amount lớn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tham lam (luôn lấy xu lớn nhất)
Xem `greedy` ở phần test: `[1,3,4]`, `6` cho `3` thay vì `2`. **Vì sao sai:** chọn xu lớn nhất có thể "phá" khả năng ghép chính xác bằng ít xu hơn. **Code đúng:** DP/BFS xét mọi xu cuối cùng.

### Lỗi 2: Khởi tạo `dp` bằng `0`
#### Code sai

In [20]:
def sai_khoi_tao_0(coins, amount):
    dp = [0] * (amount + 1)                          # SAI: các tổng chưa tạo được bị coi là cần 0 xu
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount]


for coins, amount in [([1, 2, 5], 11), ([2], 3)]:
    print(coins, amount, "đúng:", sol.coinChange(coins, amount), "| sai:", sai_khoi_tao_0(coins, amount))

[1, 2, 5] 11 đúng: 3 | sai: 0
[2] 3 đúng: -1 | sai: 0


**Vì sao sai:** `min(0, ...)` luôn `0`, mọi ô thành `0`. Phải khởi tạo `inf` cho các ô chưa biết (chỉ `dp[0] = 0`).

### Lỗi 3: Quên đổi `inf` thành `-1`
#### Code sai

In [21]:
def sai_quen_tru_1(coins, amount):
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount]                                # SAI: có thể trả inf thay vì -1


print("[2], 3 đúng:", sol.coinChange([2], 3), "| sai:", sai_quen_tru_1([2], 3))

[2], 3 đúng: -1 | sai: inf


**Vì sao sai:** không tạo được thì `dp[amount]` vẫn là `inf`; đề yêu cầu `-1`.

### Lỗi 4: Không kiểm tra `c <= a` (chỉ số âm quay vòng)
#### Code sai

In [22]:
def sai_chi_so_am(coins, amount):
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            dp[a] = min(dp[a], dp[a - c] + 1)            # SAI: a - c có thể âm -> đọc dp[-k] (phần tử cuối!)
    return -1 if dp[amount] == INF else dp[amount]


for coins, amount in [([2, 5], 3), ([3, 5], 4)]:
    print(coins, amount, "đúng:", sol.coinChange(coins, amount), "| sai:", sai_chi_so_am(coins, amount))

[2, 5] 3 đúng: -1 | sai: 2
[3, 5] 4 đúng: -1 | sai: -1


**Vì sao sai:** chỉ số âm trong Python **quay vòng** về cuối danh sách; kết quả sai mà không báo lỗi. Luôn kiểm tra `c <= a`.

### Lỗi 5: Nhầm với "đếm số cách" (Coin Change II)
#### Code sai

In [23]:
def sai_dem_so_cach(coins, amount):
    ways = [1] + [0] * amount
    for c in coins:
        for a in range(c, amount + 1):
            ways[a] += ways[a - c]                       # SAI: đây là số cách, không phải số xu ít nhất
    return ways[amount]


print("[1,2,5], 11 đúng (số xu ít nhất):", sol.coinChange([1, 2, 5], 11), "| số cách:", sai_dem_so_cach([1, 2, 5], 11))

[1,2,5], 11 đúng (số xu ít nhất): 3 | số cách: 11


**Vì sao sai:** Coin Change II (LeetCode 518) hỏi **số tổ hợp**; bài này hỏi **số xu ít nhất**: dùng `min`, không dùng cộng dồn.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**ít nhất / nhiều nhất số phần tử để đạt tổng**, mỗi phần tử dùng nhiều lần" → **DP kiểu ba lô đầy đủ (unbounded knapsack)**.
- Công thức: `dp[a] = min(dp[a - c] + 1)`; khởi tạo `dp[0] = 0`, còn lại `inf`; cuối cùng `inf → -1`.
- Cũng có thể coi là **BFS trên đồ thị số tiền** (số lớp = số xu).
- **Tham lam** chỉ dùng khi hệ xu "chính tắc" — nếu đề không đảm bảo thì dùng DP.
- Luôn kiểm tra `c <= a` trước khi đọc `dp[a - c]`.
- Họ hàng: **Coin Change II** (518, số cách), **Perfect Squares** (279), **Minimum Cost For Tickets** (983), **Combination Sum IV** (377), **Word Break** (139).
- Thử: `amount = 0`, xu lớn hơn `amount`, `[1,3,4]`/`6`, không tạo được.

## 15. Biến thể của bài

1. **Coin Change II** (LeetCode 518): đếm **số tổ hợp** tạo `amount`.
2. **Perfect Squares** (279): xu là các số chính phương — cùng thuật toán.
3. **Truy vết** các xu đã dùng.
4. **Mỗi xu dùng tối đa một lần** (0/1 knapsack): duyệt `a` giảm dần.
5. **Số xu nhiều nhất** (đổi `min` thành `max` và `-inf`).
#### Code demo

In [24]:
def coin_change_ways(coins, amount):
    ways = [1] + [0] * amount
    for c in coins:
        for a in range(c, amount + 1):
            ways[a] += ways[a - c]
    return ways[amount]


def num_squares(n):
    squares = [i * i for i in range(1, int(n ** 0.5) + 1)]
    return sol.coinChange(squares, n)


def coin_change_with_coins(coins, amount):
    dp = [0] + [INF] * amount
    last = [0] * (amount + 1)
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a], last[a] = dp[a - c] + 1, c
    if dp[amount] == INF:
        return -1, []
    used, a = [], amount
    while a:
        used.append(last[a])
        a -= last[a]
    return dp[amount], used


print(coin_change_ways([1, 2, 5], 5), num_squares(12), num_squares(13), coin_change_with_coins([1, 2, 5], 11))
# 4 3 2 (3, [1, 5, 5]) hoặc tương tự

4 3 2 (3, [1, 5, 5])


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần | thử mọi xu cuối cùng | `O(n^S)` | `O(S)` | hiểu bài toán |
| 2. Đệ quy có nhớ | `lru_cache` theo số tiền | `O(S·n)` | `O(S)` | viết nhanh từ công thức |
| 3. DP bottom-up | điền `dp[0..amount]` | `O(S·n)` | `O(S)` | **bản nộp chuẩn** |
| 4. BFS | lớp = số xu | `O(S·n)` | `O(S)` | dừng sớm khi đáp án nhỏ |
| 5. DP + `break` | sắp xếp, cắt tỉa xu lớn | `O(S·n)` | `O(S)` | tối ưu hằng số |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `Solution().coinChange([1], 5)`; (b) `Solution().coinChange([3], 7)`; (c) `Solution().coinChange([2, 5], 0)`; (d) `min(INF, 4) + 1`.

**Bài 2 (tính bằng tay).** Lập bảng `dp[0..8]` cho `coins = [2, 3]` và cho biết `dp[8]`, `dp[1]`.

**Bài 3 (điền TODO).** Hoàn thiện `coin_todo(coins, amount)` theo Cách 3.
#### Code demo

In [25]:
def coin_todo(coins, amount):
    # TODO 1: dp = [0] + [INF] * amount
    # TODO 2: for a in range(1, amount + 1): for c in coins:
    # TODO 3: nếu c <= a: dp[a] = min(dp[a], dp[a - c] + 1)
    # TODO 4: trả -1 nếu dp[amount] == INF, ngược lại dp[amount]
    pass


# Tự làm xong rồi mới mở test:
# for coins, amount, expected in examples:
#     assert coin_todo(coins, amount) == expected

**Bài 4 (tự viết lại).** Viết `coin_change_bfs2(coins, amount)` bằng BFS, **không nhìn lại** Cách 4; đối chiếu với `oracle`.

**Bài 5 (biến thể / mini challenge).** Viết `min_coins_each_used_at_most_once(coins, amount)` (mỗi **loại** xu dùng tối đa **một** lần) bằng DP 0/1 (duyệt `a` giảm dần); và `max_coins(coins, amount)` tìm **số xu nhiều nhất** tạo `amount` (hoặc `-1`). Đối chiếu bằng brute force nhỏ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: chỉ xu `1` thì cần đúng `amount` xu; không tạo được thì `-1`. Bài 2: dùng công thức `dp[a] = 1 + min(dp[a-2], dp[a-3])`. Bài 5: 0/1 thì duyệt `a` từ lớn xuống nhỏ để mỗi xu dùng một lần.
- **Mức 2 (vừa):** Bài 1: `5`, `-1`, `0`, `5`. Bài 2: `dp = [0, inf, 1, 1, 2, 2, 2, 3, 3]`; `dp[8] = 3` (`3+3+2`), `dp[1]` không tạo được. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 4: hàng đợi các tổng, `visited`, đếm lớp. Bài 5: 0/1: `for c in coins: for a in range(amount, c - 1, -1): dp[a] = min(dp[a], dp[a - c] + 1)`; `max_coins`: đổi `min` thành `max` với `-inf`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert Solution().coinChange([1], 5) == 5 and Solution().coinChange([3], 7) == -1
assert Solution().coinChange([2, 5], 0) == 0 and min(INF, 4) + 1 == 5

# Bài 2
b2 = [0] + [INF] * 8
for a in range(1, 9):
    for c in (2, 3):
        if c <= a:
            b2[a] = min(b2[a], b2[a - c] + 1)
assert b2 == [0, INF, 1, 1, 2, 2, 2, 3, 3] and Solution().coinChange([2, 3], 8) == 3


# Bài 3
def coin_todo_ans(coins, amount):
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return -1 if dp[amount] == INF else dp[amount]


for coins, amount, expected in examples:
    assert coin_todo_ans(coins, amount) == expected


# Bài 4
def coin_change_bfs2(coins, amount):
    if amount == 0:
        return 0
    seen, frontier, steps = {0}, [0], 0
    while frontier:
        steps += 1
        nxt_frontier = []
        for a in frontier:
            for c in coins:
                n2 = a + c
                if n2 == amount:
                    return steps
                if n2 < amount and n2 not in seen:
                    seen.add(n2)
                    nxt_frontier.append(n2)
        frontier = nxt_frontier
    return -1


rng = random.Random(23)
for _ in range(300):
    coins = rng.sample(range(1, 10), rng.randint(1, 4))
    amount = rng.randint(0, 20)
    assert coin_change_bfs2(coins, amount) == oracle(coins, amount)


# Bài 5
def min_coins_each_used_at_most_once(coins, amount):
    dp = [0] + [INF] * amount
    for c in coins:
        for a in range(amount, c - 1, -1):
            dp[a] = min(dp[a], dp[a - c] + 1)
    return -1 if dp[amount] == INF else dp[amount]


def max_coins(coins, amount):
    NEG = float("-inf")
    dp = [0] + [NEG] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] != NEG:
                dp[a] = max(dp[a], dp[a - c] + 1)
    return -1 if dp[amount] == NEG else dp[amount]


def brute_01(coins, amount):
    best = None
    for mask in range(1 << len(coins)):
        chosen = [coins[i] for i in range(len(coins)) if mask >> i & 1]
        if sum(chosen) == amount and (best is None or len(chosen) < best):
            best = len(chosen)
    return -1 if best is None else best


for _ in range(300):
    coins = [rng.randint(1, 9) for _ in range(rng.randint(1, 6))]
    amount = rng.randint(0, 25)
    assert min_coins_each_used_at_most_once(coins, amount) == brute_01(coins, amount)
assert max_coins([1, 2, 5], 11) == 11 and max_coins([2], 3) == -1
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (DP bottom-up)

```python
class Solution:
    def coinChange(self, coins: List[int], amount: int) -> int:
        dp = [0] + [float("inf")] * amount
        for a in range(1, amount + 1):
            for c in coins:
                if c <= a:
                    dp[a] = min(dp[a], dp[a - c] + 1)
        return -1 if dp[amount] == float("inf") else dp[amount]
```

**Checklist**

- [ ] `dp[0] = 0`, các ô còn lại `inf` (không phải `0`)?
- [ ] `dp[a] = min(dp[a], dp[a - c] + 1)` với `c <= a` (không đọc chỉ số âm)?
- [ ] Cuối cùng đổi `inf` thành `-1`; `amount = 0` trả `0`?
- [ ] Không dùng tham lam; biết phản ví dụ `[1,3,4]`/`6`?
- [ ] Phân biệt với **đếm số cách** (Coin Change II)?
- [ ] Đã thử xu lớn hơn `amount`, không tạo được, `amount = 10^4`?
- [ ] Đã đối chiếu với brute force liệt kê số lượng từng loại xu?
- [ ] Skeleton đúng `class Solution` / `def coinChange(self, coins: List[int], amount: int) -> int`?